# Phenotype exploration -- data science for funsies

Reads directly from `MODELING_TABLE_DIR` (`residualize_phenotypes.ipynb`'s `prepare_modeling_tables()` output -- one TSV per phenotype, already pulled/filtered/covariate-joined) -- no BigQuery calls anywhere in this notebook, so it's cheap to rerun as often as you like. Four things:

1. Rank phenotypes by number of observations (post plausible-range + keep-list filter)
2. Raw vs. rank-inverse-normal-transformed density, per phenotype -- does the skew transform actually do anything?
3. Raw vs. residualized density + "sound statistics" (N retained, R², post-residual skewness) across the nested covariate-set staircase (`base` -> `base_pcs` -> `base_pcs_zip3` -> `base_pcs_zip3_ses`), reusing `residualize_lib.R`'s real `residualize_phenotype()` -- not a reimplementation
4. Pairwise correlations between phenotypes (on the rank-inverse-normal-transformed scale, so a skewed phenotype doesn't distort the correlation the way it would on the raw scale)

**Subsampling policy:** every skewness/R²/N-observation statistic below is computed on the *full* filtered table for that phenotype -- only plot rendering and the correlation join subsample to `MAX_PLOT_N`, since a density plot or scatter of 250k+ points buys nothing over 20k and costs a lot of render time. Same output-hygiene note as every other real-data notebook here: this pulls person-level rows into memory (never printed directly, only plotted/aggregated), so clear outputs / run `nbstripout` before committing, per root `README.md`.

## Setup

In [ ]:
required_pkgs <- c("dplyr", "readr", "e1071", "ggplot2")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(ggplot2)
source("../../scripts/local/residualize_lib.R")   # skew_summary()/residualize_phenotype()/build_covariate_sets() -- shared with residualize_phenotypes.ipynb, not reimplemented here

theme_set(theme_minimal(base_size = 12))

## Inputs

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
PHENOTYPE_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, "data", "02_phenotype")
MODELING_TABLE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "modeling_tables")   # residualize_phenotypes.ipynb's prepare_modeling_tables() output

PHENO_LIST_PATH <- "../../docs/phenotype_list.tsv"
pc_cols <- paste0("PC", 1:5)   # top 5 only, same convention as residualize_phenotypes.ipynb

MAX_PLOT_N <- 20000   # subsample cap for plotting/joins only -- see subsampling policy above
RANDOM_SEED <- 0
set.seed(RANDOM_SEED)

subsample_df <- function(df, n = MAX_PLOT_N) {
  if (nrow(df) > n) df[sample(nrow(df), n), ] else df
}

## Load modeling tables

Tries every `phenotype_name` in `docs/phenotype_list.tsv`, not just the ones with a confirmed `concept_id` -- `hip_circumference`/`waist_hip_ratio` are still `UNCONFIRMED` as of writing, so their tables won't exist yet, but file-existence is a simpler and more robust "is this ready" check here than replicating `residualize_phenotypes.ipynb`'s `concept_id != "UNCONFIRMED"` filter logic (which doesn't even cleanly cover `waist_hip_ratio`'s `"UNCONFIRMED,UNCONFIRMED"` shape). Missing tables are skipped with a `message()`, same error-tolerant convention as `prepare_modeling_tables()`.

In [ ]:
pheno_list <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))

load_modeling_table <- function(name) {
  path <- file.path(MODELING_TABLE_DIR, paste0(name, ".tsv"))
  if (!file.exists(path)) {
    message(sprintf("Skipping '%s': no modeling table at %s", name, path))
    return(NULL)
  }
  read_tsv(path, show_col_types = FALSE)
}

pheno_tables <- setNames(lapply(pheno_list$phenotype_name, load_modeling_table), pheno_list$phenotype_name)
pheno_tables <- pheno_tables[!sapply(pheno_tables, is.null)]

sprintf("Loaded %d of %d phenotypes' modeling tables", length(pheno_tables), nrow(pheno_list))

## 1. Rank by number of observations

N after `filter_plausible_range()` + the keep-list restriction -- i.e. the same N `prepare_modeling_tables()` actually wrote to each phenotype's table, not a fresh count.

In [ ]:
n_obs_table <- tibble(
  phenotype_name = names(pheno_tables),
  n = sapply(pheno_tables, nrow)
) %>% arrange(desc(n))

n_obs_table

ggplot(n_obs_table, aes(x = reorder(phenotype_name, n), y = n)) +
  geom_col(fill = "#3d5a80") +
  coord_flip() +
  labs(title = "Phenotypes ranked by number of observations", x = NULL,
       y = "N (post plausible-range filter + keep-list restriction)")

## 2. Skewness transform: raw vs. invnorm

`phenotype__invnorm` is already in each modeling table (written by `add_transformed_variant()` inside `prepare_modeling_tables()`) -- no retransforming here, just comparing what's already there. `skew_summary()` (from `residualize_lib.R`) on the full table; the density plot below subsamples per phenotype for rendering only.

In [ ]:
skew_summary_table <- bind_rows(lapply(names(pheno_tables), function(name) {
  df <- pheno_tables[[name]]
  bind_rows(
    skew_summary(df$phenotype, "raw"),
    skew_summary(df$phenotype__invnorm, "invnorm")
  ) %>% mutate(phenotype_name = name, .before = 1)
}))
skew_summary_table

In [ ]:
density_long <- bind_rows(lapply(names(pheno_tables), function(name) {
  df <- subsample_df(pheno_tables[[name]])
  bind_rows(
    tibble(phenotype_name = name, variant = "raw", value = df$phenotype),
    tibble(phenotype_name = name, variant = "invnorm", value = df$phenotype__invnorm)
  )
}))

ggplot(density_long, aes(x = value, fill = variant)) +
  geom_density(alpha = 0.5, color = NA) +
  facet_wrap(~ phenotype_name, scales = "free", ncol = 4) +
  scale_fill_manual(values = c(raw = "#e07a5f", invnorm = "#3d5a80")) +
  labs(title = "Raw vs. rank-inverse-normal-transformed density, per phenotype", x = NULL, y = NULL) +
  theme(legend.position = "top", strip.text = element_text(size = 8))

## 3. Residualization: raw vs. residualized, and R² across the covariate staircase

Reuses `residualize_lib.R`'s real `residualize_phenotype()` -- same function `run_residualization_from_tables()` calls, not a reimplementation. All-NA covariate columns (zip3/SES not always populated for every phenotype's cohort) are dropped from that phenotype's model first, same guard `run_residualization_from_tables()` uses, rather than letting `lm()` silently produce 0 complete cases.

In [ ]:
covariate_sets <- build_covariate_sets(pc_cols)

usable_covariates <- function(df, cols) {
  cols[sapply(cols, function(col) !all(is.na(df[[col]])))]
}

fit_full_model <- function(df) {
  residualize_phenotype(df, "phenotype", "sex_at_birth", usable_covariates(df, covariate_sets$base_pcs_zip3_ses))
}

full_model_results <- lapply(pheno_tables, fit_full_model)

residual_stats_table <- bind_rows(lapply(names(full_model_results), function(name) {
  r <- full_model_results[[name]]
  tibble(
    phenotype_name = name,
    n_input = r$n_input,
    n_retained = r$n_retained,
    pct_retained = round(100 * r$n_retained / r$n_input, 1),
    r_squared = round(r$r_squared, 4),
    skewness_post = round(e1071::skewness(r$data$phenotype_norm, na.rm = TRUE), 3)
  )
})) %>% arrange(desc(r_squared))

residual_stats_table

In [ ]:
residual_density_long <- bind_rows(lapply(names(full_model_results), function(name) {
  df <- subsample_df(full_model_results[[name]]$data)
  bind_rows(
    tibble(phenotype_name = name, variant = "raw", value = df$phenotype),
    tibble(phenotype_name = name, variant = "residualized (full model)", value = df$phenotype_norm)
  )
}))

ggplot(residual_density_long, aes(x = value, fill = variant)) +
  geom_density(alpha = 0.5, color = NA) +
  facet_wrap(~ phenotype_name, scales = "free", ncol = 4) +
  scale_fill_manual(values = c("raw" = "#e07a5f", "residualized (full model)" = "#81b29a")) +
  labs(title = "Raw vs. residualized (base_pcs_zip3_ses) density, per phenotype", x = NULL, y = NULL) +
  theme(legend.position = "top", strip.text = element_text(size = 8))

### R² across the nested covariate-set staircase

`build_covariate_sets()`'s staircase (`base` -> `base_pcs` -> `base_pcs_zip3` -> `base_pcs_zip3_ses`), refit per phenotype on the raw variant -- how much of each successive block (PCs, zip3, SES) actually buys in variance explained, not just the full model's final R² from above.

In [ ]:
staircase_r2 <- bind_rows(lapply(names(pheno_tables), function(name) {
  df <- pheno_tables[[name]]
  bind_rows(lapply(names(covariate_sets), function(covset_name) {
    cols <- usable_covariates(df, covariate_sets[[covset_name]])
    if (length(cols) == 0) return(NULL)
    r <- residualize_phenotype(df, "phenotype", "sex_at_birth", cols)
    tibble(phenotype_name = name, covariate_set = covset_name, r_squared = r$r_squared)
  }))
}))
staircase_r2$covariate_set <- factor(staircase_r2$covariate_set, levels = names(covariate_sets))

ggplot(staircase_r2, aes(x = covariate_set, y = r_squared, group = 1)) +
  geom_line(color = "#3d5a80") +
  geom_point(color = "#3d5a80", size = 2) +
  facet_wrap(~ phenotype_name, ncol = 4) +
  labs(title = "R² as covariates are added (nested staircase, raw variant)", x = NULL, y = expression(R^2)) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1), strip.text = element_text(size = 8))

## 4. Correlations between phenotypes

On the invnorm scale (already rank-based, so a skewed phenotype doesn't distort the correlation the way the raw scale would). Join cost is bounded regardless of cohort size: samples `MAX_PLOT_N` person_ids from the phenotype with the most observations, restricts every other phenotype's table to that same sample before joining -- `cor(..., use = "pairwise.complete.obs")` still only uses whichever pairs of phenotypes both have a non-NA value for a given person, so phenotypes with very different cohorts (e.g. `waist_circumference`) aren't penalized for not overlapping perfectly with the sampled ids.

In [ ]:
largest_pheno <- n_obs_table$phenotype_name[1]
sampled_ids <- pheno_tables[[largest_pheno]]$person_id
if (length(sampled_ids) > MAX_PLOT_N) sampled_ids <- sample(sampled_ids, MAX_PLOT_N)

wide_invnorm <- Reduce(function(acc, name) {
  col <- pheno_tables[[name]] %>%
    filter(person_id %in% sampled_ids) %>%
    transmute(person_id, !!name := phenotype__invnorm)
  full_join(acc, col, by = "person_id")
}, names(pheno_tables), tibble(person_id = sampled_ids))

cor_mat <- cor(wide_invnorm %>% select(-person_id), use = "pairwise.complete.obs")

cor_long <- as.data.frame(as.table(cor_mat)) %>%
  rename(phenotype_x = Var1, phenotype_y = Var2, correlation = Freq)

ggplot(cor_long, aes(x = phenotype_x, y = phenotype_y, fill = correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "#3d5a80", mid = "white", high = "#e07a5f", midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Pairwise phenotype correlations (invnorm scale)", x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

## Summary

`n_obs_table` -- which phenotypes are actually high-completeness vs. barely-usable (compare against `waist_circumference`'s known N=22 problem). `skew_summary_table` -- which phenotypes needed the invnorm transform at all (near-0 raw skewness means it didn't do much) vs. which it clearly fixed. `residual_stats_table`/the staircase plot -- which covariates are actually buying variance explained per phenotype (a flat staircase past `base` means zip3/SES aren't adding much for that trait) and whether `pct_retained` looks reasonable (a low value means the 5-SD post-residual trim is cutting hard, worth a second look). `cor_long`/the heatmap -- expected clusters (e.g. HDL/LDL/total cholesterol/triglycerides should correlate; height and hemoglobin shouldn't) are a sanity check on the pipeline more than a discovery in themselves at this stage.